## Entrenamiento en Paralelo y Evaluación de Escalabilidad

### SVM con Multiprocessing
A continuación, se entrena el modelo distribuyendo el dataset de manera dinámica en diferentes números de subprocesos para medir el impacto en los tiempos de ejecución y evaluar el comportamiento de la exactitud promedio.

In [ ]:
# Función objetivo para los procesos hijos
def train_subset(data):
    X_sub, y_sub = data
    model = SVC(
        kernel='rbf',
        C=10,
        gamma='scale',
        random_state=42
    )
    model.fit(X_sub, y_sub)
    return model

tiempos = []
accura = []
NUM_PROCESOS = 7

for i in range(2, NUM_PROCESOS + 1):
    # Dividir dataset dinámicamente según el número de procesos 'i'
    indices = np.array_split(np.arange(len(X_train)), i)
    data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

    start_par = time.time()

    # Crear un Pool con 'i' procesos configurados
    with mp.Pool(processes=i) as pool:
        models = pool.map(train_subset, data_splits)

    t_par = time.time() - start_par

    # Evaluar individualmente todos los modelos obtenidos
    acc_ind = []
    for _, model in enumerate(models):
        pred_par = model.predict(X_test)
        acc_par = accuracy_score(y_test, pred_par)
        acc_ind.append(acc_par)

    # Calcular la exactitud promedio de esta iteración
    acc_promedio = np.mean(acc_ind)

    print(f"Procesadores activos: {i}")
    print(f"Tiempo paralelo: {t_par:.2f} segundos")
    print(f"Accuracy promedio: {acc_promedio:.4f}\n")

    tiempos.append(t_par)
    accura.append(acc_promedio)